In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

print("1. Iniciando Spark...")
spark = SparkSession.builder \
    .appName("NYC_GreenTaxi_Pipeline") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("2. Leyendo datos crudos (Bronze)...")
# Asegúrate de que el nombre del CSV coincida con el que tienes en tu carpeta (con guiones bajos o medios)
df_bronze_trips = spark.read.parquet("green_tripdata_*.parquet")
df_bronze_zones = spark.read.csv("taxi-zone-lookup.csv", header=True, inferSchema=True) 

print("3. Limpiando y transformando datos (Silver)...")
df_silver = df_bronze_trips.filter(
    (F.col("trip_distance") > 0) & 
    (F.col("fare_amount") > 0) & 
    (F.col("passenger_count") > 0)
).dropna(subset=["PULocationID", "DOLocationID"])

df_silver = df_silver \
    .withColumn("mes", F.month("lpep_pickup_datetime")) \
    .withColumn("dia_semana", F.dayofweek("lpep_pickup_datetime")) \
    .withColumn("hora", F.hour("lpep_pickup_datetime"))

df_silver_enriched = df_silver.join(
    df_bronze_zones, 
    df_silver.PULocationID == df_bronze_zones.LocationID, 
    "left"
).withColumnRenamed("Borough", "pickup_borough") \
 .withColumnRenamed("Zone", "pickup_zone")

print("4. Generando archivos CSV para Power BI (Gold)...")
df_silver_enriched.createOrReplaceTempView("taxis_silver")

# Query 1: Demanda
spark.sql("""
    SELECT dia_semana, hora, COUNT(*) as total_viajes
    FROM taxis_silver GROUP BY dia_semana, hora ORDER BY dia_semana, hora
""").coalesce(1).write.mode("overwrite").csv("powerbi_demanda", header=True)

# Query 2: Zonas
spark.sql("""
    SELECT pickup_borough as distrito, pickup_zone as zona, COUNT(*) as total_viajes, ROUND(AVG(total_amount), 2) as ingreso_promedio_usd
    FROM taxis_silver GROUP BY pickup_borough, pickup_zone HAVING total_viajes > 500 ORDER BY ingreso_promedio_usd DESC
""").coalesce(1).write.mode("overwrite").csv("powerbi_zonas", header=True)

# Query 3: Pagos
spark.sql("""
    SELECT 
        CASE 
            WHEN payment_type = 1 THEN 'Tarjeta de Crédito'
            WHEN payment_type = 2 THEN 'Efectivo'
            WHEN payment_type = 3 THEN 'Sin Cargo'
            WHEN payment_type = 4 THEN 'Disputa'
            ELSE 'Otro'
        END as tipo_pago,
        COUNT(*) as cantidad_viajes
    FROM taxis_silver GROUP BY tipo_pago
""").coalesce(1).write.mode("overwrite").csv("powerbi_pagos", header=True)

# Query 4: Dispersión
spark.sql("""
    SELECT trip_distance as distancia_millas, total_amount as tarifa_total, hora
    FROM taxis_silver WHERE trip_distance < 30 AND total_amount < 150
""").sample(fraction=0.05, seed=42).coalesce(1).write.mode("overwrite").csv("powerbi_dispersion", header=True)

print("¡Proceso completado! Los 4 archivos CSV ya están listos en tus carpetas.")

1. Iniciando Spark...
2. Leyendo datos crudos (Bronze)...
3. Limpiando y transformando datos (Silver)...
4. Generando archivos CSV para Power BI (Gold)...
¡Proceso completado! Los 4 archivos CSV ya están listos en tus carpetas.
